# 📞 Telecom Churn Prediction Pipeline
## Leakage-Free ML Pipeline with Class Imbalance Handling

---

### 🎯 Problem Statement (Hinglish)

Aap ek telecom company mein data scientist ho. Aapko ek **churn prediction model** banana hai jo predict kare ki customer service chhodega ya nahi.

**Dataset Info:**
- Numerical features hain
- Target column: `churn` (0 = no churn, 1 = churn)
- **Class imbalance**: 73% no-churn, 27% churn

**Aapke 5 tasks hain:**
1. Scaling se **PEHLE** data split karo
2. `StandardScaler` ko **sirf training** pe fit karo
3. **Experiment 1**: Training set pe SMOTE apply karo
4. **Experiment 2**: `class_weight='balanced'` use karo
5. Dono models ko **unmodified validation set** pe evaluate karo

---

## 📦 Step 1: Import Libraries

### Hinglish Explanation:
Pehle saari zaroori libraries import karte hain. Notice — `imblearn` (imbalanced-learn) ek **separate library** hai, sklearn ka part nahi. Agar installed nahi hai, toh terminal mein chalao:
```
pip install imbalanced-learn
```

**Har library ka role:**
- `pandas`, `numpy` → data handling
- `train_test_split` → data ko parts mein todna
- `StandardScaler` → features ko same scale pe laana
- `LogisticRegression` → humara ML model
- `classification_report` → detailed evaluation (precision, recall, f1)
- `SMOTE` → minority class ke synthetic samples banane ke liye

In [ ]:
import pandas as pd
import numpy as np

# Sklearn ke main components
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Imbalanced data ke liye special library
from imblearn.over_sampling import SMOTE

# Reproducibility ke liye seed set karte hain
np.random.seed(42)

print("✅ Saari libraries successfully import ho gayi!")

## 🏗️ Step 2: Sample Dataset Banao

### Hinglish Explanation:
Real assignment mein aapko CSV diya jaayega — `pd.read_csv('telecom.csv')` se load kar lena.

Yahaan demo ke liye **synthetic data** bana rahe hain — 4000 customers ka data, 73-27 ratio mein churn distribution. `p=[0.73, 0.27]` parameter exactly woh ratio enforce karta hai.

In [ ]:
# Sample dataset banate hain (4000 customers)
n = 4000

sample_df = pd.DataFrame({
    "tenure_months": np.random.randint(1, 72, size=n),          # Service tenure
    "monthly_charges": np.random.uniform(20, 120, size=n),      # Monthly bill
    "total_charges": np.random.uniform(100, 8000, size=n),      # Total billed
    "support_calls": np.random.poisson(2, size=n),              # Support calls count
    "churn": np.random.choice([0, 1], size=n, p=[0.73, 0.27])  # Target: 73-27 split
})

print("📊 Dataset shape:", sample_df.shape)
print("\n📋 First 5 rows:")
sample_df.head()

## 🔍 Step 3: Class Imbalance Check Karo

### Hinglish Explanation:
Modeling se **pehle hamesha** class distribution check karo. Agar imbalance hai (jaise 73-27), toh accuracy alone reliable metric nahi hoti — `classification_report` use karna padega.

**Yaad rakho:** Agar model sirf majority class (0) predict kare, toh accuracy = **73%** — lekin model bekaar hai! Isliye imbalance ko handle karna important hai.

In [ ]:
# Class distribution check karte hain
class_counts = sample_df['churn'].value_counts()
class_percent = sample_df['churn'].value_counts(normalize=True) * 100

print("🎯 Class Distribution:")
print(f"   Class 0 (No Churn): {class_counts[0]} samples ({class_percent[0]:.1f}%)")
print(f"   Class 1 (Churn):    {class_counts[1]} samples ({class_percent[1]:.1f}%)")
print(f"\n⚠️  Imbalance Ratio: {class_counts[0] / class_counts[1]:.2f} : 1")
print("\n💡 Note: 73-27 split confirms data is imbalanced — special handling needed!")

## ✂️ Step 4: Features aur Target Alag Karo

### Hinglish Explanation:
- **X** (features) = saare columns **except** target
- **y** (target) = sirf `churn` column

Ye standard ML practice hai. `drop()` se target column hata dete hain X se.

In [ ]:
target_col = "churn"

# Features (X) aur target (y) separate karo
X = sample_df.drop(columns=[target_col])
y = sample_df[target_col]

print(f"✅ X shape (features): {X.shape}")
print(f"✅ y shape (target):   {y.shape}")
print(f"\n📋 Feature columns: {list(X.columns)}")

## 🔀 Step 5: Data Split — Train / Validation / Test

### Hinglish Explanation (MOST IMPORTANT STEP!):

Ye step **sabse critical** hai leakage prevent karne ke liye. Data ko **3 parts** mein todenge:
- **Train (70%)** → model seekhne ke liye
- **Validation (15%)** → hyperparameter tuning, model selection ke liye
- **Test (15%)** → final unbiased evaluation ke liye (untouched!)

### Splitting Strategy:
Hum **2-step split** karenge:
1. Pehle 15% test alag karo → bachega 85% (train + val)
2. Fir 85% ko train aur val mein todo

### ⭐ `stratify=y` kyun important hai?

`stratify=y` ensure karta hai ki **har split mein 73-27 ratio maintain rahe**. Bina iske, by chance ek split mein 90% class-0 aa sakta hai — galat evaluation.

### 🧮 0.176 ka math:
Hume final split chahiye 70-15-15. Pehle 15% test nikala (85% bacha). Ab 85% ka kitna % chahiye validation ke liye taki final mein 15% mile?
→ **15 / 85 ≈ 0.176** ✅

In [ ]:
# Step 5a: Pehle TEST set alag karo (15%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.15,        # 15% test ke liye
    random_state=42,        # Reproducibility ke liye
    stratify=y              # Class ratio maintain karne ke liye
)

# Step 5b: Baki 85% ko TRAIN aur VALIDATION mein todo
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.176,        # 15/85 ≈ 0.176 (final 15% validation ke liye)
    random_state=42,
    stratify=y_train_val    # Yahan bhi stratify zaroori
)

print("📊 Split Sizes:")
print(f"   Train:      {X_train.shape[0]} samples ({X_train.shape[0]/n*100:.1f}%)")
print(f"   Validation: {X_val.shape[0]} samples ({X_val.shape[0]/n*100:.1f}%)")
print(f"   Test:       {X_test.shape[0]} samples ({X_test.shape[0]/n*100:.1f}%)")

print("\n🎯 Class distribution verify karte hain (stratify check):")
print(f"   Train churn %: {y_train.mean()*100:.1f}%")
print(f"   Val churn %:   {y_val.mean()*100:.1f}%")
print(f"   Test churn %:  {y_test.mean()*100:.1f}%")
print("\n✅ Sab splits mein ~27% churn ratio maintain hua — stratify successful!")

## ⚖️ Step 6: Scaling — Leakage-Free Tarika

### Hinglish Explanation (CRITICAL — Most Common Mistake!):

Ye step **sabse zyada mistake** wala hai. Dhyaan se padho!

### ❌ GALAT Tarika (Leakage hoga):
```python
# Pehle full data scale kar di — LEAKAGE!
scaler.fit_transform(X)
X_train, X_val = train_test_split(X)
```

### ✅ SAHI Tarika (Leakage-free):
```python
# Pehle split, fir sirf train pe fit
scaler.fit_transform(X_train)  # fit + transform sirf train pe
scaler.transform(X_val)        # val pe sirf transform!
```

### Logic samjho:
`StandardScaler` data ka **mean** aur **standard deviation** seekhta hai. Agar tum full data pe fit karoge, toh validation/test ka mean/std bhi seekh lega — matlab model ne validation ko "dekh liya" before training. Ye **leakage** hai.

**Rule yaad rakho:** `fit` sirf train pe, `transform` sab pe.

In [ ]:
# Scaler initialize karo
scaler = StandardScaler()

# ✅ SAHI: fit_transform SIRF training data pe
X_train_scaled = scaler.fit_transform(X_train)

# ✅ SAHI: validation aur test pe SIRF transform (fit nahi!)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("✅ Scaling complete — leakage prevent ho gayi!")
print(f"\n📊 Training data statistics (after scaling):")
print(f"   Mean (should be ~0): {X_train_scaled.mean():.4f}")
print(f"   Std (should be ~1):  {X_train_scaled.std():.4f}")

print(f"\n📊 Validation data statistics:")
print(f"   Mean (NOT exactly 0): {X_val_scaled.mean():.4f}")
print(f"   Std (NOT exactly 1):  {X_val_scaled.std():.4f}")
print("\n💡 Val ka mean exactly 0 nahi hai — kyunki scaler train pe fit hua tha.")
print("   Agar mean=0 aata, toh leakage ka sign hota!")

## 🧪 Step 7: Experiment 1 — SMOTE (Synthetic Minority Over-sampling)

### Hinglish Explanation:

**SMOTE kya karta hai?**
Minority class (churn=1) ke samples kam hain. SMOTE in samples ke **synthetic versions** banata hai — interpolation ke through nearby minority samples ke beech mein naye points generate karta hai.

**Example:** Agar 2000 No-Churn aur 740 Churn hain training mein, SMOTE 1260 fake-but-realistic Churn samples banayega, taaki final mein 2000 vs 2000 ho jaaye — balanced!

### ⚠️ GOLDEN RULE:
**SMOTE sirf TRAINING data pe lagao!** Validation/Test pe **kabhi nahi**.

**Kyun?** Validation ka purpose hai real-world performance check karna. Real world mein synthetic data hota hi nahi. Agar val pe SMOTE lagaya, toh fake data pe evaluate kar rahe ho — meaningless results.

In [ ]:
print("🔬 EXPERIMENT 1: SMOTE\n" + "="*50)

# Step 7a: SMOTE initialize karo
smote = SMOTE(random_state=42)

# Step 7b: SMOTE SIRF training data pe apply karo (val/test pe NAHI!)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"📊 Before SMOTE (Training data):")
print(f"   Class 0: {(y_train==0).sum()} samples")
print(f"   Class 1: {(y_train==1).sum()} samples")

print(f"\n📊 After SMOTE (Training data):")
print(f"   Class 0: {(y_train_smote==0).sum()} samples")
print(f"   Class 1: {(y_train_smote==1).sum()} samples")
print("\n✅ Dono classes ab balanced hain — 50-50 ratio!")

# Step 7c: Logistic Regression model train karo SMOTE data pe
model_smote = LogisticRegression(random_state=42, max_iter=1000)
model_smote.fit(X_train_smote, y_train_smote)

# Step 7d: Predict on ORIGINAL validation set (NOT modified!)
y_pred_smote = model_smote.predict(X_val_scaled)

print("\n📈 Experiment 1 Results (SMOTE):")
print("-" * 50)
print(classification_report(y_val, y_pred_smote, target_names=['No Churn', 'Churn']))
print(f"Accuracy: {accuracy_score(y_val, y_pred_smote):.4f}")

## 🧪 Step 8: Experiment 2 — `class_weight='balanced'`

### Hinglish Explanation:

**`class_weight='balanced'` kya karta hai?**
Ye SMOTE ki tarah naya data **nahi banata**. Ye bas model ko bolta hai — *"minority class ke errors ko zyada important maan, zyada penalty de."*

### Mathematical Logic:
Weights automatically calculate hote hain inverse proportional to class frequency:
- Class 0 (73%): weight ≈ low
- Class 1 (27%): weight ≈ high

Matlab agar model class-1 ko misclassify kare, toh penalty zyada lagegi. Isse model class-1 pe zyada attention deta hai.

### SMOTE vs class_weight — Difference:
| SMOTE | class_weight='balanced' |
|-------|------------------------|
| Naye synthetic samples banata hai | Existing samples ko alag importance deta hai |
| Data size badh jaata hai | Data size same rehta hai |
| Slower (preprocessing step) | Faster (training mein hi handle) |
| Sometimes overfits on synthetic data | More stable, conservative approach |

In [ ]:
print("🔬 EXPERIMENT 2: class_weight='balanced'\n" + "="*50)

# Step 8a: Model with class_weight='balanced'
# Notice: SMOTE NAHI use ho raha. Original X_train_scaled aur y_train use ho rahe hain
model_cw = LogisticRegression(
    class_weight='balanced',   # ⭐ Ye magic parameter hai
    random_state=42,
    max_iter=1000
)

# Step 8b: Train on original (unmodified) training data
model_cw.fit(X_train_scaled, y_train)

# Step 8c: Predict on validation set
y_pred_cw = model_cw.predict(X_val_scaled)

print("📈 Experiment 2 Results (class_weight='balanced'):")
print("-" * 50)
print(classification_report(y_val, y_pred_cw, target_names=['No Churn', 'Churn']))
print(f"Accuracy: {accuracy_score(y_val, y_pred_cw):.4f}")

## 📊 Step 9: Dono Experiments Compare Karo

### Hinglish Explanation:

Imbalanced data mein **accuracy dekhna alone bekaar** hai. Hume **recall** aur **f1-score** pe focus karna chahiye, especially **minority class (Churn=1)** ke liye.

**Key Metrics samjho:**
- **Precision (Churn)**: Jab model bola "Churn karega", kitni baar sahi tha?
- **Recall (Churn)**: Real Churners mein se kitne model ne pakde?
- **F1-Score**: Precision aur Recall ka balance

**Business Context:** Telecom company ke liye **recall** zyada important hai — kyunki churners ko miss karna zyada costly hai (customer ja chuka). False positive (galti se churner bolna) cheaper hai (sirf retention offer waste).

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Dono experiments ke metrics calculate karo
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (Churn)', 'Recall (Churn)', 'F1-Score (Churn)'],
    'Experiment 1: SMOTE': [
        accuracy_score(y_val, y_pred_smote),
        precision_score(y_val, y_pred_smote),
        recall_score(y_val, y_pred_smote),
        f1_score(y_val, y_pred_smote)
    ],
    'Experiment 2: class_weight': [
        accuracy_score(y_val, y_pred_cw),
        precision_score(y_val, y_pred_cw),
        recall_score(y_val, y_pred_cw),
        f1_score(y_val, y_pred_cw)
    ]
})

# Format to 4 decimal places
results['Experiment 1: SMOTE'] = results['Experiment 1: SMOTE'].round(4)
results['Experiment 2: class_weight'] = results['Experiment 2: class_weight'].round(4)

print("🏆 FINAL COMPARISON TABLE\n" + "="*60)
print(results.to_string(index=False))

print("\n💡 Note: Synthetic data ke saath actual results vary karte hain.")
print("   Real telecom data pe, dono techniques alag tarah perform karti hain.")

## 🔄 Step 10: Pura Pipeline Ek Function Mein

### Hinglish Explanation:

Production code mein, sab cheez ek **function** mein hona chahiye — reusable aur clean. Niche pura pipeline ek function mein wrap kar diya. Ye exam mein **as-is** likh sakte ho.

In [ ]:
def build_churn_pipeline(df: pd.DataFrame, target_col: str = "churn"):
    """
    Complete leakage-free churn prediction pipeline.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe with features + target column
    target_col : str
        Name of target column (default: 'churn')
    
    Returns:
    --------
    dict with both trained models and their validation scores
    """
    
    # Step 1: Features aur target separate
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Step 2: 3-way split with stratification
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.176, 
        random_state=42, stratify=y_train_val
    )
    
    # Step 3: Scaling — fit only on train (LEAKAGE PREVENTION)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    
    # ===== EXPERIMENT 1: SMOTE =====
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
    
    model_smote = LogisticRegression(random_state=42, max_iter=1000)
    model_smote.fit(X_train_res, y_train_res)
    y_pred_smote = model_smote.predict(X_val_scaled)
    
    print("=== Experiment 1: SMOTE ===")
    print(classification_report(y_val, y_pred_smote))
    
    # ===== EXPERIMENT 2: class_weight='balanced' =====
    model_cw = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
    model_cw.fit(X_train_scaled, y_train)
    y_pred_cw = model_cw.predict(X_val_scaled)
    
    print("=== Experiment 2: class_weight='balanced' ===")
    print(classification_report(y_val, y_pred_cw))
    
    return {
        'model_smote': model_smote,
        'model_cw': model_cw,
        'scaler': scaler,
        'predictions_smote': y_pred_smote,
        'predictions_cw': y_pred_cw
    }

# Function call karo
results = build_churn_pipeline(sample_df, target_col="churn")
print("\n✅ Pipeline complete!")

## 🎯 Key Takeaways (Yaad Rakho!)

### 🔑 5 Golden Rules:

1. **Split FIRST, Transform LATER** — Hamesha. Pehle data todo, fir process karo.

2. **`fit` only on Training** — `fit_transform` sirf X_train pe. Validation/Test pe sirf `transform`. Kabhi nahi `fit_transform` validation pe!

3. **SMOTE only on Training Set** — Validation aur Test untouched rakhne hain. Real-world simulation karni hai.

4. **`stratify=y` for Classification** — Class ratio maintain karne ke liye. Specially imbalanced data mein.

5. **`classification_report` for Imbalanced Data** — Sirf accuracy mat dekho. Precision, Recall, F1 dekho — especially minority class ka.

### ❌ Top 3 Mistakes to Avoid:

- ❌ `scaler.fit_transform(X)` **before** train_test_split → **Leakage!**
- ❌ SMOTE on full data **before** split → **Leakage!**
- ❌ Accuracy as sole metric for imbalanced data → **Misleading!**

### 📚 Concepts Used:
- ✅ Train-Validation-Test split (70-15-15)
- ✅ Stratified sampling
- ✅ StandardScaler with leakage prevention
- ✅ SMOTE (Synthetic Minority Over-sampling)
- ✅ class_weight='balanced'
- ✅ classification_report for evaluation

---

### 💡 Bonus: Real-World Tips

- **Test set sirf END mein** use karo — final model evaluation ke liye. Hyperparameter tuning ke liye validation use karo.
- **Both experiments compare karke** decide karo — kabhi SMOTE better, kabhi class_weight. Data dependent hai.
- **Production mein** scaler aur model dono ko **save** karna (pickle/joblib) — same scaler future predictions pe use karna hai.

---

**🎓 All the best for your assignment!**